In [29]:
# autoreload
%load_ext autoreload
%autoreload 2

from openai import OpenAI

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index, VectorSearch
#from toyaikit.llm import OpenAIClient
#from toyaikit.tools import Tools
#from toyaikit.chat import IPythonChatInterface
#from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

import sys
sys.path.append("../")
#from rag_helper import RAGBase
from embedder import Embedder

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
def text_search(index, query, num_results=5, filter_dict=None, boost_dict=None):
    """
    Search the index for a given query.

    Args:
        index: The minsearch Index instance.
        query (str): The search query.
        num_results (int): The number of results to return. Default is 5.
        filter_dict (dict): A dictionary of filters to apply to the search. Default is None.
        boost_dict (dict): A dictionary of fields to boost in the search. Default is None

    Returns:
        list: A list of search results.
    """
    return index.search(
        query,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=num_results
    )

In [31]:
def vector_search(index, embedding, query, num_results=5):
    """
    Search the index for a given query.

    Args:
        index: The minsearch VectorSearch instance.
        embedding: The embedding model instance.
        query (str): The search query.
        num_results (int): The number of results to return. Default is 5.

    Returns:
        list: A list of search results.
    """
    query_vector = embedding.encode(query)
    return index.search(query_vector, num_results=num_results)


In [78]:
def rrf(result_lists, k=60, num_results=5):
    """
    Fuse multiple ranked result lists using Reciprocal Rank Fusion (RRF).

    Each document gets a fused score computed as:

        score(doc) = sum(1 / (k + rank))

    where `rank` is 1-based within each input list and `k` controls how
    strongly top-ranked results are favored.

    Args:
        result_lists (list[list[dict]]): Ranked lists of documents.
            Each document is expected to contain at least:
            - "filename" (str)
            - "start" (int)
        k (int, optional): RRF constant. Higher values reduce rank impact.
            Defaults to 60.
        num_results (int, optional): Number of fused results to return.
            Defaults to 5.

    Returns:
        list[dict]: Top fused documents, ordered by descending fused score.
    """
    if num_results <= 0:
        return []
    if k < 0:
        raise ValueError("k must be >= 0")

    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results, start=1):  # 1-based rank
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            docs[key] = doc

    ranked_items = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [docs[key] for key, _ in ranked_items[:num_results]]

### 1. Retrieve the data from the GitHub repository and parse it into a list of documents

In [32]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [33]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [34]:
# Number of documents
len(documents)

72

## 2. Test embeddings

In [35]:
# use downloaded Xenova/all-MiniLM-L6-v2 model to embed documents

embed = Embedder(path="../models/Xenova/all-MiniLM-L6-v2/")

In [36]:
Query = "How does approximate nearest neighbor search work?"

In [37]:
v = embed.encode(Query)

In [38]:
# The embedder returns a vector of 384 numbers. What's the first value (v[0])?
v[0]

np.float64(-0.02058203437252893)

In [39]:
[doc for doc in documents if doc.get("filename") == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0]

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [40]:
page_content = [doc for doc in documents if doc.get("filename") == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0].get("content")

In [41]:
v1=embed.encode(page_content)

In [42]:
# Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?
v.dot(v1)

np.float64(0.36107027225589694)

## 3. Chunking and embeding

In [43]:
# Create chunks of the documents for reindexing
chunks = chunk_documents(documents, size=2000, step=1000)

In [44]:
# Embed the chunks with encode batch
X = embed.encode_batch([chunk.get("content") for chunk in chunks])

In [45]:
#score the Q1 query against all chunks. Which file does the highest-scoring chunk belong to (its filename)?
scores = X.dot(v)

In [46]:
# Top 5 score indexes
top5 =scores.argsort()[-5:][::-1]
top5

array([ 94,  14, 162,  85, 253])

In [47]:
# Top 5 scores
top5_scores = scores[top5]
top5_scores

array([0.64890177, 0.55103463, 0.40656071, 0.40618198, 0.40610594])

In [48]:
# The filename of the highest-scoring chunk
chunks[top5[0]]["filename"]


'02-vector-search/lessons/07-sqlitesearch-vector.md'

## 4. Vector search with MinSearch

In [63]:
## Create a Vector index with minsearch
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, chunks)

In [64]:
query = "What metric do we use to evaluate a search engine?"

In [65]:
results = vector_search(vindex, embed, query, num_results=5)

In [66]:
# Which file is the filename of the first result?
results[0]["filename"]

'04-evaluation/lessons/05-search-metrics.md'

### 4.1 Create text index with minsearch

In [72]:
query = "How do I store vectors in PostgreSQL?"

In [73]:
# Instantiate the index
index = Index(
    text_fields = ["content"],
    keyword_fields = ["filename"],)
# Fit the index with the documents
index.fit(chunks)

### 4.2 Compare text and vector search results

In [74]:
results_text = text_search(index, query, num_results=5)

In [75]:
results_vector = vector_search(vindex, embed, query, num_results=5)

In [76]:
#Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
vector_filenames = {result["filename"] for result in results_vector}
text_filenames = {result["filename"] for result in results_text}

vector_only_filenames = vector_filenames - text_filenames
vector_only_filenames

{'02-vector-search/lessons/08-pgvector.md'}

## 5. Hybrid search with MinSearch

In [79]:
query = "How do I give the model access to tools?"
results_text = text_search(index, query, num_results=5)
results_vector = vector_search(vindex, embed, query, num_results=5)

results_hybrid = rrf([results_text, results_vector], k=60, num_results=5)

In [83]:
def top_filename(results):
    return results[0]["filename"] if results else None

top_text = top_filename(results_text)
top_vector = top_filename(results_vector)
top_hybrid = top_filename(results_hybrid)

display({
    "text_top": top_text,
    "vector_top": top_vector,
    "hybrid_top": top_hybrid,
    "text==hybrid": top_text == top_hybrid,
    "vector==hybrid": top_vector == top_hybrid,
})

{'text_top': '01-agentic-rag/lessons/14-agentic-loop.md',
 'vector_top': '01-agentic-rag/lessons/01-intro.md',
 'hybrid_top': '01-agentic-rag/lessons/13-function-calling.md',
 'text==hybrid': False,
 'vector==hybrid': False}